只要没落地就有。首先横向匀速运动，同一帧肯定初始横坐标大的横坐标更大；其次纵向做匀加速运动，正方向取上，同rnd下，初始横坐标大的向上初速度越大，由于加速度都一样，所以到同一帧的每帧内的，也是初始横坐标大的速度越大，所以纵坐标也就越大；最后int对<=保持序关系，所以判定坐标也满足这个条件

小鬼不落地时显然有：
- 同rnd同一帧，坐标越大的巨人投出的小鬼横坐标越大，纵坐标越小（纵向偏移越大）
- 同巨人坐标同一帧，rnd越大的巨人投出的小鬼纵坐标越大（纵向偏移越小），横坐标相同
（以上坐标均为浮点坐标）
由于int对<=保持序关系，所以上面的结论对int坐标也成立



基于我现在的代码，给定时间t，我的点群是x0[t][i], y0[t][i]

所以同一帧，小鬼坐标的边界

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']  # 任选其一存在即可
mpl.rcParams['axes.unicode_minus'] = False

In [ ]:
x0_df = pd.read_csv("x0.csv").drop(columns=["index"])
y0_df = pd.read_csv("y0.csv").drop(columns=["index"])
x1_df = pd.read_csv("x1.csv").drop(columns=["index"])
y1_df = pd.read_csv("y1.csv").drop(columns=["index"])
x0, y0 = x0_df.values, y0_df.values # higher
x1, y1 = x1_df.values, y1_df.values # lower
# shape: (time, traj_num)
n_frame, n_traj = x0.shape

In [ ]:
# x0[t][i] <= x1[t][i]
# y0[t][i] <= y0[t][i]
# 

In [ ]:
for t in range(n_frame):
    fig, ax = plt.subplots(figsize=(8, 6))
    plt.scatter(x0[t], y0[t], color="r", alpha=0.1, s=2, marker=".")
    plt.scatter(x1[t], y1[t], color="b", alpha=0.1, s=2, marker=".")
    ax.invert_yaxis()
    plt.xlim(0, 800)
    plt.ylim(0, 600)
    plt.show()
    break

In [ ]:
import matplotlib.animation as animation
from IPython.display import HTML

fig, ax = plt.subplots(figsize=(16, 8), dpi=120)
# ax.set_xlim(0, 800)
# ax.set_ylim(0, 600)
ax.invert_yaxis()              
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Imp Trajectories (animated)")
ax.set_aspect("equal", adjustable="box")
ax.grid(True, alpha=0.25)

# 初始散点（用小点 s=1）
scat0 = ax.scatter(x0[0], y0[0], c="r", alpha=0.12, s=1, marker=".")
scat1 = ax.scatter(x1[0], y1[0], c="b", alpha=0.12, s=1, marker=".")

# 可选：画代表性曲线（第0条轨迹）
line0_0, = ax.plot(x0[:, 0], y0[:, 0], color="r", lw=0.8, alpha=0.6)
line1_0, = ax.plot(x1[:, 0], y1[:, 0], color="b", lw=0.8, alpha=0.6)
# 可选：画代表性曲线（第-1条轨迹）
line0_1, = ax.plot(x0[:, -1], y0[:, -1], color="r", lw=0.8, alpha=0.6)
line1_1, = ax.plot(x1[:, -1], y1[:, -1], color="b", lw=0.8, alpha=0.6)
def update(frame):
    # 更新散点
    offs0 = np.column_stack((x0[frame], y0[frame]))
    offs1 = np.column_stack((x1[frame], y1[frame]))
    scat0.set_offsets(offs0)
    scat1.set_offsets(offs1)
    # 如果想实时只画到当前帧的代表轨迹，可做下面两行；否则可注释掉
    line0_0.set_data(x0[:frame+1, 0], y0[:frame+1, 0])
    line1_0.set_data(x1[:frame+1, 0], y1[:frame+1, 0])
    line0_1.set_data(x0[:frame+1, -1], y0[:frame+1, -1])
    line1_1.set_data(x1[:frame+1, -1], y1[:frame+1, -1])
    return scat0, scat1, line0_0, line1_0, line0_1, line1_1

fps = 10  # 每秒帧数
interval_ms = 1000 // fps  # 每帧间隔（毫秒），可调

anim = animation.FuncAnimation(fig, update, frames=n_frame, interval=interval_ms, blit=False)

# 保存为 mp4
anim.save("traj.mp4", fps=fps, dpi=150, writer="ffmpeg")

# 在 notebook 中显示
# HTML(anim.to_jshtml())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
plt.plot(x1[:, 0], y1[:, 0], label="100.0f", color="b")
plt.plot(x0[:, 0], y0[:, 0], label="0.0f", color="r")
ax.invert_yaxis()
plt.legend()
plt.show()

In [ ]:
def plot_envelope(ax, color="C0", label=None, draw_samples=True, sample_num=30, alpha_traj=0.08):
    # 包络: 每一帧的上下左右边界
    x_min_on_frame = x1.min(axis=1)
    x_max_on_frame = x0.max(axis=1)
    x_min = x0.min(axis=0)
    x_max = x.max(axis=0)
    y_min = y.min(axis=0)
    y_max = y.max(axis=0)

    # 用 fill_betweenx 按 y 方向填充“横向包络”
    ax.fill_betweenx(
        y=(y_min + y_max) / 2,
        x1=x_min,
        x2=x_max,
        color=color,
        alpha=0.25,
        label=label
    )

    # 也把上下边界线画出来，便于观察
    ax.plot(x_min, y_min, color=color, lw=1)
    ax.plot(x_max, y_max, color=color, lw=1)

    # 可选: 叠加部分原始轨迹
    if draw_samples:
        step = max(1, n_traj // sample_num)
        for i in range(0, n_traj, step):
            ax.plot(x[i], y[i], color=color, alpha=alpha_traj, lw=0.8)


In [ ]:

fig, ax = plt.subplots(figsize=(8, 6), dpi=140)

plot_envelope(ax, x0, y0, color="tab:blue", label="group 0")
plot_envelope(ax, x1, y1, color="tab:red", label="group 1")

ax.set_title("Imp Trajectory Envelope")
ax.set_xlabel("x")
ax.set_ylabel("y")

# 关键: 屏幕坐标系，左上角为原点，向下为 y 正方向
ax.invert_yaxis()

ax.set_aspect("equal", adjustable="box")
ax.grid(True, alpha=0.25)
ax.legend()
plt.show()